In [1]:
import os
import sys
from pathlib import Path
ROOT = Path("../..").resolve()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sys.path.insert(0, str(ROOT / "experiments" / "lfm"))
os.getcwd()

'/home/user/projects/SoftStairs-QAT'

In [2]:
from lfm_model import (
    LFMCausalTrainer,
    fit_lfm_model,
    get_text_dataloaders,
    compute_snapshot_steps,
)

/home/user/projects/SoftStairs-QAT/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0901 13:53:47.642000 94807 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 13:53:47.674000 94807 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [ ]:
from datasets import load_dataset

raw = load_dataset("wikitext", "wikitext-2-raw-v1")
train_n = len(raw["train"].filter(lambda x: len(x["text"].strip()) > 0))
val_n = len(raw["validation"].filter(lambda x: len(x["text"].strip()) > 0))

DATA_FRACTION = 0.45  

LIMIT_TRAIN = int(DATA_FRACTION * train_n)   
LIMIT_VAL = int(DATA_FRACTION * val_n)         

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Fri Aug 28 13:04:59 2026).


In [4]:
from softstairs_qat.utils import ReproducibilityManager, configure_logging
from loguru import logger

ReproducibilityManager().set_seed(42)
paths = configure_logging(name="lfm_qat", log_dir="experiments/lfm/logs")
logger.info("Log file: {}", paths.log_file)

MODEL_ID = "LiquidAI/LFM2-350M"
DATASET = "wikitext"
BATCH_SIZE = 4
MAX_SEQ_LEN = 256
LIMIT_TRAIN = 5000
LIMIT_VAL = 500
LR = 2e-5
N_EPOCHS = 20

TORCH_QAT_BITS = 8
TORCH_QAT_GROUP_SIZE = 32
BASELINE_RUN = f"baseline-torch-qat-int{TORCH_QAT_BITS}b-{N_EPOCHS}e"

2026-09-01 14:29:14 | INFO     | __main__:<module>:6 - Log file: experiments/lfm/logs/lfm_qat_20260901_142914.log


In [5]:
from softstairs_qat import QuantizationConfig

SS_TYPE = "standard" 

def get_qconfig(strategy, t_start, total_steps, n_bits=8, ss_type=SS_TYPE):
    return QuantizationConfig(
        n_bits=n_bits,
        normalized=True,
        type=ss_type,
        t_scheduler_strategy=strategy,
        t_start=t_start,
        t_end=1e-4,
        n_steps=total_steps,
    )

In [6]:
train_loader, val_loader, _ = get_text_dataloaders(
    dataset=DATASET,
    model_id=MODEL_ID,
    batch_size=BATCH_SIZE,
    max_length=MAX_SEQ_LEN,
    limit_train=LIMIT_TRAIN,
    limit_val=LIMIT_VAL,
)
steps_per_epoch = len(train_loader)
logger.info("train samples={}, val samples={}, steps/epoch={}", LIMIT_TRAIN, LIMIT_VAL, steps_per_epoch)

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Fri Aug 28 13:04:59 2026).
2026-09-01 14:29:21 | INFO     | __main__:<module>:10 - train samples=5000, val samples=500, steps/epoch=1250


# Baseline

In [7]:
import torch
from torchao.quantization import IntxWeightOnlyConfig, quantize_
from torchao.quantization.qat import QATConfig
from torchao.quantization.granularity import PerGroup, PerAxis


def get_torch_qat_config(n_bits=8, group_size=32):
    weight_dtype = torch.int8 if n_bits == 8 else torch.int4
    if group_size is None:
        return IntxWeightOnlyConfig(
            weight_dtype=weight_dtype,
            granularity=PerAxis(0),
        )
    return IntxWeightOnlyConfig(
        weight_dtype=weight_dtype,
        granularity=PerGroup(group_size),
    )


def run_baseline_nb(
    n_epochs=N_EPOCHS,
    n_bits=TORCH_QAT_BITS,
    group_size=TORCH_QAT_GROUP_SIZE,
    run_name=None,
):
    torch_qconfig = get_torch_qat_config(n_bits, group_size)
    if run_name is None:
        run_name = f"baseline-torch-qat-int{n_bits}b-{n_epochs}e"

    logger.info("Starting baseline run={} bits={} group_size={}", run_name, n_bits, group_size)

    trainer = LFMCausalTrainer(
        model_id=MODEL_ID,
        lr=LR,
        weight_decay=0.0,
        run_name=run_name,
        torch_qconfig=torch_qconfig,
        snapshot_steps=compute_snapshot_steps(n_epochs, steps_per_epoch),
    )

    fit_lfm_model(trainer, train_loader, val_loader, max_epochs=n_epochs)
    logger.info("Finished baseline run={}", run_name)
    return trainer, torch_qconfig

In [8]:
trainer, torch_qconfig = run_baseline_nb(n_epochs=N_EPOCHS)

2026-09-01 14:29:28 | INFO     | __main__:run_baseline_nb:30 - Starting baseline run=baseline-torch-qat-int8b-20e bits=8 group_size=32
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 460.44it/s]
/home/user/projects/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torch/autograd/graph.py:979: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:833.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[baseline-torch-qat-int8b-20e] epoch 1/20 loss=3.0217 val_loss=3.3047 ppl=27.24
[baseline-torch-qat-int8b-20e] epoch 2/20 loss=1.9724 val_loss=3.4871 ppl=32.69
[baseline-torch-qat-int8b-20e] epoch 3/20 loss=1.1086 val_loss=3.9374 ppl=51.29
[baseline-torch-qat-int8b-20e] epoch 4/20 loss=0.5828 val_loss=4.3126 ppl=74.64
[baseline-torch-qat-int8b-20e] epoch 5/20 loss=0.3412 val_loss=4.6187 ppl=101.36
[baseline-torch-qat-int8b-20e] epoch 6/20 loss=0.2579 val_loss=4.7961 ppl=121.04
[baseline-torch-qat-int8b-20e] epoch 7/20 loss=0.2257 val_loss=5.0022 ppl=148.73
[baseline-torch-qat-int8b-20e] epoch 8/20 loss=0.2006 val_loss=5.0652 ppl=158.42
[baseline-torch-qat-int8b-20e] epoch 9/20 loss=0.1888 val_loss=5.1655 ppl=175.12
[baseline-torch-qat-int8b-20e] epoch 10/20 loss=0.1729 val_loss=5.2556 ppl=191.64
[baseline-torch-qat-int8b-20e] epoch 11/20 loss=0.1599 val_loss=5.4173 ppl=225.26
[baseline-torch-qat-int8b-20e] epoch 12/20 loss=0.1520 val_loss=5.5228 ppl=250.34
[baseline-torch-qat-int8b-20e

2026-09-01 15:45:08 | INFO     | __main__:run_baseline_nb:42 - Finished baseline run=baseline-torch-qat-int8b-20e


[baseline-torch-qat-int8b-20e] epoch 20/20 loss=0.1222 val_loss=6.0051 ppl=405.51


# LFM running

In [ ]:
def run_experiment_nb(
    n_epochs,
    strategy,
    t_start,
    n_bits=8,
    weight_decay=0.0,
    dataset="wikitext",
    model_id=MODEL_ID,
    batch_size=BATCH_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    limit_train=LIMIT_TRAIN,
    limit_val=LIMIT_VAL,
):
    train_loader, val_loader, _ = get_text_dataloaders(
        dataset=dataset,
        model_id=model_id,
        batch_size=batch_size,
        max_length=max_seq_len,
        limit_train=limit_train,
        limit_val=limit_val,
    )

    steps_per_epoch = len(train_loader)
    total_steps = n_epochs * steps_per_epoch
    qconfig = get_qconfig(strategy, t_start, total_steps, n_bits, ss_type=SS_TYPE)
    run_name = f"{strategy}-{t_start}-{n_bits}b-{SS_TYPE}-{n_epochs}e-wd{weight_decay}"

    logger.info(
        "Starting run={} dataset={} steps/epoch={} total_steps={}",
        run_name, dataset, steps_per_epoch, total_steps,
    )

    model = LFMCausalTrainer(
        model_id=model_id,
        lr=LR,
        weight_decay=weight_decay,
        run_name=run_name,
        snapshot_steps=compute_snapshot_steps(n_epochs, steps_per_epoch),
        qconfig=qconfig,
    )

    fit_lfm_model(model, train_loader, val_loader, max_epochs=n_epochs)
    logger.info("Finished run={}", run_name)
    return run_name

In [8]:
run_experiment_nb(
    n_epochs=1,
    strategy="linear",
    t_start=0.9,
    n_bits=8,
    weight_decay=0.0,
)

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/user/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Fri Aug 28 13:04:59 2026).
2026-08-28 13:44:51 | INFO     | __main__:run_experiment_nb:28 - Starting run=linear-0.9-8b-1e-wd0.0 dataset=wikitext steps/epoch=1250 total_steps=1250
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 511.33it/s]
/home/user/projects/SoftStairs-QAT/.venv/lib/python3.10/site-packages/torch/autograd/graph.py:979: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:833.)
  return Variable._execution_engine.run_backward(  # Call

[linear-0.9-8b-1e-wd0.0] epoch 1/1 loss=4.8064 val_loss=14.6457 ppl=2293765.88 t=0.0001


'linear-0.9-8b-1e-wd0.0'